# 📘 MCDI502 – Semana 3 · Formativa 3: Preparación para el Taller II
**Gestión de Datos y Tecnologías Big Data · Actividad grupal, sin nota (0%)**

Flujo: Instalación → Configuración → Ingesta OpenMeteo (5 registros) → Ingesta AQICN (5 registros) → Transformaciones → Consultas SQL → Integración básica → Resumen final.

> ⚠️ Ejecutar las celdas **en orden** y esperar a que cada una termine antes de ejecutar la siguiente.

# PARTE 1: Instalación de Dependencias

In [ ]:
# 1.1 Instalar las librerías necesarias (puede tardar 2-3 minutos)
!pip install -q requests
!pip install -q pandas
!pip install -q pyspark
!pip install -q findspark

print("✅ Instalación completada")

# PARTE 2: Configuración del Entorno

In [ ]:
# 2.1 Importar librerías y definir constantes del entorno
import time
import requests
import pandas as pd
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Número de registros por fuente en la formativa (en la sumativa serán 20)
N_REGISTROS = 5

print("✅ Entorno configurado")

In [ ]:
# 2.2 Crear la SparkSession (modo local, todos los cores)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("FormativaS3_MCDI502") \
    .getOrCreate()

print("✅ SparkSession creada")
print(f"Nombre de la aplicación: {spark.sparkContext.appName}")

# PARTE 3: Ingesta OpenMeteo
Basada en `BaseEjemploOpenMeteo.ipynb`. Obtiene 5 registros meteorológicos y crea un DataFrame de Spark.

In [ ]:
# 3.1 Parámetros de la API OpenMeteo (Viña del Mar, Chile)
LATITUD = -33.02457
LONGITUD = -71.55183

# Variables requeridas por la guía
VARIABLES = [
    "temperature_2m",         # Temperatura a 2 metros (°C)
    "relative_humidity_2m",   # Humedad relativa (%)
    "precipitation",          # Precipitación (mm)
    "wind_speed_10m",         # Velocidad del viento (km/h)
    "uv_index"                # Índice UV
]

def obtener_datos():
    """Devuelve el bloque 'current' de OpenMeteo o {} si hay error."""
    joined_vars = ",".join(VARIABLES)
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={LATITUD}&longitude={LONGITUD}"
        f"&current={joined_vars}&timezone=America/Santiago"
    )
    try:
        respuesta = requests.get(url, timeout=30)
        if respuesta.status_code == 200:
            return respuesta.json().get("current", {})
    except requests.RequestException as e:
        print(f"⚠️ Error de conexión: {e}")
    return {}

print("✅ Parámetros de API OpenMeteo configurados")

In [ ]:
# 3.2 Capturar 5 registros (10 s entre consultas, como en el notebook base)
historico = []
intentos, MAX_INTENTOS = 0, N_REGISTROS * 3

while len(historico) < N_REGISTROS and intentos < MAX_INTENTOS:
    intentos += 1
    datos_actuales = obtener_datos()
    tiempo_actual = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    if datos_actuales:
        fila = {"tiempo": tiempo_actual}
        for var in VARIABLES:
            valor = datos_actuales.get(var)
            fila[var] = float(valor) if valor is not None else None
        historico.append(fila)
        print(f"Registro {len(historico)}/{N_REGISTROS} capturado")

    if len(historico) < N_REGISTROS:
        time.sleep(10)

assert len(historico) == N_REGISTROS, "No se lograron capturar los 5 registros de OpenMeteo"
print("✅ Datos obtenidos correctamente de OpenMeteo")

In [ ]:
# 3.3 Crear DataFrame de Spark
df_pandas_om = pd.DataFrame(historico)
df_openmeteo = spark.createDataFrame(df_pandas_om)

print(f"✅ DataFrame OpenMeteo: {df_openmeteo.count()} registros")
df_openmeteo.show(truncate=False)
print("✅ Se muestran los 5 registros en la tabla")
df_openmeteo.printSchema()

# PARTE 4: Ingesta AQICN
Basada en `BaseEjemploAQICN.ipynb`. Obtiene 5 registros de calidad del aire (PM2_5, PM10, O3, NO2).

In [ ]:
# 4.1 Parámetros de la API AQICN
token = "demo"          # token de prueba del notebook base
city = "santiago"
url_aqicn = f"http://api.waqi.info/feed/{city}/?token={token}"

print("✅ Parámetros de API AQICN configurados")
print(f"Ciudad: {city} | Token: {token}")

In [ ]:
# 4.2 Capturar 5 registros (1 s entre consultas)
records = []
intentos, MAX_INTENTOS = 0, N_REGISTROS * 4

while len(records) < N_REGISTROS and intentos < MAX_INTENTOS:
    intentos += 1
    try:
        data = requests.get(url_aqicn, timeout=30).json()
    except (requests.RequestException, ValueError) as e:
        print(f"⚠️ Error de conexión/JSON: {e}")
        time.sleep(1)
        continue

    if data.get("status") == "ok":
        iaqi = data["data"]["iaqi"]
        fila = {
            "PM2_5": iaqi.get("pm25", {}).get("v"),
            "PM10":  iaqi.get("pm10", {}).get("v"),
            "O3":    iaqi.get("o3", {}).get("v"),
            "NO2":   iaqi.get("no2", {}).get("v"),
        }
        # Solo se aceptan registros con los 4 contaminantes (evita nulos en el promedio)
        if all(v is not None for v in fila.values()):
            records.append(fila)
            print(f"Registro {len(records)}/{N_REGISTROS} capturado")
        else:
            print(f"⚠️ Registro incompleto descartado: {fila}")

    if len(records) < N_REGISTROS:
        time.sleep(1)

assert len(records) == N_REGISTROS, "No se lograron capturar los 5 registros de AQICN"
print("✅ Datos obtenidos correctamente de AQICN")

In [ ]:
# 4.3 Crear DataFrame de Spark
df_pandas_aq = pd.DataFrame(records).astype(float)
df_aqicn = spark.createDataFrame(df_pandas_aq)

print(f"✅ DataFrame AQICN: {df_aqicn.count()} registros")
df_aqicn.show(truncate=False)
print("✅ Se muestran los 5 registros en la tabla")
df_aqicn.printSchema()

# PARTE 5: Transformaciones
- `temperatura_fahrenheit` = (Celsius × 9/5) + 32, redondeada a 2 decimales
- `promedio_contaminantes` = (PM2_5 + PM10 + O3 + NO2) / 4, redondeado a 2 decimales

In [ ]:
# 5.1 Columna calculada 1: temperatura_fahrenheit
df_openmeteo = df_openmeteo.withColumn(
    "temperatura_fahrenheit",
    F.round(F.col("temperature_2m") * 9 / 5 + 32, 2)
)
print("✅ Columna temperatura_fahrenheit creada")
df_openmeteo.select("temperature_2m", "temperatura_fahrenheit").show(truncate=False)
print("✅ Se muestran los valores en Fahrenheit")

In [ ]:
# 5.2 Columna calculada 2: promedio_contaminantes
df_aqicn = df_aqicn.withColumn(
    "promedio_contaminantes",
    F.round((F.col("PM2_5") + F.col("PM10") + F.col("O3") + F.col("NO2")) / 4, 2)
)
print("✅ Columna promedio_contaminantes creada")
df_aqicn.show(truncate=False)
print("✅ Se muestran los valores de promedio")

# PARTE 6: Consultas SQL

In [ ]:
# 6.1 Registrar vistas temporales
df_openmeteo.createOrReplaceTempView("openmeteo_data")
df_aqicn.createOrReplaceTempView("aqicn_data")
print("✅ Vistas openmeteo_data y aqicn_data registradas")

In [ ]:
# 6.2 Consulta 1: Estadísticas de Temperatura
print("Consulta 1: Estadísticas de Temperatura")
spark.sql("""
SELECT
    MAX(temperature_2m) AS temp_maxima,
    MIN(temperature_2m) AS temp_minima,
    ROUND(AVG(temperature_2m), 2) AS temp_promedio
FROM openmeteo_data
""").show()

# 6.3 Consulta 2: Promedio de Contaminantes
print("Consulta 2: Promedio de Contaminantes")
spark.sql("""
SELECT
    ROUND(AVG(promedio_contaminantes), 2) AS promedio_general
FROM aqicn_data
""").show()

# 6.4 Consulta 3: Humedad relativa promedio
print("Consulta 3: Humedad relativa promedio")
spark.sql("""
SELECT
    ROUND(AVG(relative_humidity_2m), 2) AS humedad_promedio
FROM openmeteo_data
""").show()

# PARTE 7: Integración básica

In [ ]:
# 7.1 DataFrame unificado (UNION de ambas fuentes) + resumen por fuente
df_om_f = df_openmeteo.withColumn("fuente", F.lit("OpenMeteo"))
df_aq_f = df_aqicn.withColumn("fuente", F.lit("AQICN"))

# unionByName con columnas faltantes rellenas con null (esquemas distintos)
df_unificado = df_om_f.unionByName(df_aq_f, allowMissingColumns=True)

print(f"✅ DataFrame unificado: {df_unificado.count()} registros, {len(df_unificado.columns)} columnas")

resumen = (df_unificado.groupBy("fuente").count()
           .withColumnRenamed("count", "registros")
           .orderBy(F.desc("fuente")))   # OpenMeteo primero, luego AQICN
resumen.show()

# PARTE 8: Resumen final

In [ ]:
# 8.1 Resumen de lo completado
n_om, n_aq = df_openmeteo.count(), df_aqicn.count()
checks = {
    "Instalación completada": True,
    "Entorno configurado y SparkSession creada": spark is not None,
    "DataFrame OpenMeteo con 5 registros": n_om == N_REGISTROS,
    "DataFrame AQICN con 5 registros": n_aq == N_REGISTROS,
    "Columna temperatura_fahrenheit": "temperatura_fahrenheit" in df_openmeteo.columns,
    "Columna promedio_contaminantes": "promedio_contaminantes" in df_aqicn.columns,
    "3 consultas SQL ejecutadas": True,
    "Integración básica (OpenMeteo + AQICN)": df_unificado.count() == n_om + n_aq,
}
for k, v in checks.items():
    print(("✅ " if v else "❌ ") + k)

print()
if all(checks.values()):
    print("🎉 ACTIVIDAD FORMATIVA COMPLETADA")
else:
    print("⚠️ Revisar los puntos marcados con ❌ antes de cerrar")